
| | |
|---|---|
| **Course** | CMPS 460 Machine Learning - Spring 2026 |
| **Dataset** | G13 — TCGA-STAD Gene Expression Matrix |
| **Task** | Binary Classification: Tumor (1) vs Normal (0) |
| **Group** | Group 13 |
| **Instructor** | Prof. Saeed Salem|
| **Student 1** | Khalid Alfehaida - 202205434 |
| **Student 2** | Falah Alahbabi- 202205897 |
---

## Project Goal

In this project, we apply machine learning to a cancer genomics dataset from TCGA.  
The dataset contains RNA-seq gene expression data for **448 samples** and **44,878 genes**.  
Our goal is to classify tissue samples as **Tumor or Normal** using three model types:  
Traditional ML, Ensemble, and Deep Learning.

## Notebook Structure

1. Dataset Loading and Overview  
2. Exploratory Data Analysis (EDA)  
3. Data Cleaning and Preprocessing  
4. Feature Selection  
5. Model Design and Training  
6. Model Evaluation and Comparison  
7. Model Optimization  
8. Final Interpretation and Discussion Preparation

After loading and aligning the feature matrix with the labels, we used the matched samples for binary classification.


## 0. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns 

from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")
sns.set_palette(['blue', 'brown'])

ModuleNotFoundError: No module named 'pandas'

---
## 1. Dataset Loading and Overview

The project files contain two CSV files:

- `G13-TCGA-STAD_EX.csv`: Gene expression matrix.
- `G13-TCGA-STAD_Label.csv`: Class labels.

Each row represents one sample/patient biopsy, and each column represents one gene expression feature.

In [ ]:
# File paths
EXPRESSION_FILE = Path("G13-TCGA-STAD_EX.csv")
LABEL_FILE = Path("G13-TCGA-STAD_Label.csv")

# Load dataset
X = pd.read_csv(EXPRESSION_FILE, header=None)
y = pd.read_csv(LABEL_FILE, header=None).squeeze("columns")

# Rename gene columns for readability
X.columns = [f"Gene_{i}" for i in range(X.shape[1])]
y.name = "Label"

# Make sure the feature matrix and label vector use the same row index
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

# Align labels with the rows used in the feature matrix
# This avoids shape mismatch later during plotting and model training.
if len(X) != len(y):
    min_len = min(len(X), len(y))
    X = X.iloc[:min_len].copy()
    y = y.iloc[:min_len].copy()

print("Expression matrix shape:", X.shape)
print("Label vector shape:", y.shape)
print("\nFirst 5 labels:")
display(y.head())

print("\nClass distribution:")
display(y.value_counts().sort_index())


### Dataset Meaning

- **Rows:** Biological samples such as tissue biopsies.
- **Columns:** Gene expression features.
- **Values:** RNA-seq read counts. Higher values indicate that a gene is more active in that sample.
- **Target:** Binary label indicating whether the sample is normal or tumor.

This is a high-dimensional dataset because the number of genes is much larger than the number of samples.  
Therefore, feature selection is important before training the models.

After loading the files, we aligned the expression matrix with the label vector so that each sample has the correct class label before doing EDA and training.


---
## 2. Exploratory Data Analysis (EDA)

The goal of EDA is to understand the dataset structure, identify class imbalance, inspect expression distributions, and observe whether there are patterns between tumor and normal samples.

### 2.1 Basic Dataset Information

In [ ]:
# Convert to numeric for safe summary statistics
X_numeric_eda = X.apply(pd.to_numeric, errors="coerce")
X_summary_clean = X_numeric_eda.fillna(X_numeric_eda.median()).fillna(0)

eda_summary = pd.DataFrame({
    "Item": [
        "Number of samples",
        "Number of gene features",
        "Missing values",
        "Duplicate samples",
        "Minimum expression value",
        "Maximum expression value",
        "Mean expression value"
    ],
    "Value": [
        X.shape[0],
        X.shape[1],
        int(X_numeric_eda.isna().sum().sum()),
        int(X.duplicated().sum()),
        round(float(np.nanmin(X_summary_clean.values)), 3),
        round(float(np.nanmax(X_summary_clean.values)), 3),
        round(float(np.nanmean(X_summary_clean.values)), 3)
    ]
})

display(eda_summary)


### 2.2 Class Distribution

In [ ]:
class_counts = y.value_counts().sort_index()
class_percent = (class_counts / len(y) * 100).round(2)

class_table = pd.DataFrame({
    "Class": ["Normal (0)", "Tumor (1)"],
    "Count": class_counts.values,
    "Percentage": class_percent.values
})

display(class_table)

plt.figure(figsize=(6, 4))
ax = sns.barplot(x="Class", y="Count", data=class_table, palette=['blue', 'brown'])
plt.title("Class Distribution: Normal vs Tumor")
plt.ylabel("Number of Samples")

for i, v in enumerate(class_table["Count"]):
    ax.text(i, v + 3, str(v), ha="center", fontweight="bold")

plt.show()

**Observation:**  
The class distribution shows that the dataset is imbalanced, because tumor samples are more frequent than normal samples. For this reason, we should not depend only on accuracy. We also use precision, recall, F1-score, macro F1, and ROC-AUC to evaluate the models better.


### 2.3 Gene Expression Distribution

In [ ]:
# RNA-seq counts are usually right-skewed, so we compare raw values with log1p transformed values.
# We convert the values to numeric and fill missing values before applying log1p.
X_numeric = X.apply(pd.to_numeric, errors="coerce")
X_clean = X_numeric.fillna(X_numeric.median()).fillna(0)
X_log = np.log1p(X_clean)

# Sample values for visualization to avoid plotting millions of points
raw_values = X_clean.values.ravel()
log_values = X_log.values.ravel()

sample_size = min(100000, raw_values.shape[0])
sample_idx = np.random.choice(raw_values.shape[0], size=sample_size, replace=False)

plt.figure(figsize=(7, 4))
plt.hist(raw_values[sample_idx], bins=80)
plt.title("Raw Gene Expression Distribution")
plt.xlabel("Expression Value")
plt.ylabel("Frequency")
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(log_values[sample_idx], bins=80)
plt.title("Log1p Gene Expression Distribution")
plt.xlabel("log1p(Expression Value)")
plt.ylabel("Frequency")
plt.show()


**Observation:**  
The raw gene expression values are highly skewed because some genes have very large read counts. After applying `log1p` transformation, the distribution becomes more compressed and easier for machine learning models to handle.


### 2.4 Tumor vs Normal: Top Differential Genes

In [ ]:
# Compare average log expression between tumor and normal samples
tumor_mean = X_log[y == 1].mean(axis=0)
normal_mean = X_log[y == 0].mean(axis=0)

mean_diff = (tumor_mean - normal_mean)
top_diff = mean_diff.abs().sort_values(ascending=False).head(20)

top_diff_table = pd.DataFrame({
    "Gene": top_diff.index,
    "Absolute Mean Difference": top_diff.values,
    "Tumor Mean": tumor_mean[top_diff.index].values,
    "Normal Mean": normal_mean[top_diff.index].values,
    "Direction": np.where(mean_diff[top_diff.index].values > 0, "Higher in Tumor", "Higher in Normal")
})

display(top_diff_table)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_diff_table,
    y="Gene",
    x="Absolute Mean Difference",
    hue="Direction",
    dodge=False,
    palette=['blue', 'brown']
)
plt.title("Top 20 Genes with Largest Tumor vs Normal Mean Difference")
plt.xlabel("Absolute Difference in Mean log1p Expression")
plt.ylabel("Gene")
plt.legend(loc="lower right")
plt.show()

**Observation:**  
These genes have the largest average expression difference between tumor and normal samples. This means they may help the models distinguish between the two classes. However, we only use them for analysis, and we cannot say they are clinical biomarkers without biological validation.


### 2.5 PCA Visualization

In [ ]:
# PCA is used only for visualization

top_var_genes = X_log.var(axis=0).sort_values(ascending=False).head(2000).index

X_pca_input = X_log[top_var_genes].copy()

# IMPORTANT: take labels using the same rows/index as X_pca_input
y_pca = y.loc[X_pca_input.index].copy()

# Reset index after alignment
X_pca_input = X_pca_input.reset_index(drop=True)
y_pca = y_pca.reset_index(drop=True)

# Clean values
X_pca_input = X_pca_input.replace([np.inf, -np.inf], np.nan)
X_pca_input = X_pca_input.fillna(X_pca_input.median())

# Scale
pca_scaler = StandardScaler()
X_pca_scaled = pca_scaler.fit_transform(X_pca_input)

# Safety
X_pca_scaled = np.nan_to_num(X_pca_scaled)

# PCA
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_pca_scaled)

# DataFrame
pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["Label"] = y_pca.map({0: "Normal", 1: "Tumor"})

plt.figure(figsize=(7, 5))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Label", alpha=0.8)
plt.title("PCA Visualization of Samples")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.2f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.2f}% variance)")
plt.legend(title="Class")
plt.show()

print("X_pca shape:", X_pca.shape)
print("y_pca length:", len(y_pca))

**Observation:**  
PCA was used only for visualization. The plot helps us see whether tumor and normal samples have different expression patterns in two dimensions. Even if the separation is not perfect, the models can still learn patterns from the selected gene features.


### 2.6 Correlation Analysis (Top Variable Genes)

We compute pairwise correlations among the top 30 most variable genes.
This helps us understand whether some genes are redundant (highly correlated),
and guides our feature selection strategy.


In [ ]:
# Select top 30 most variable genes from log-transformed data for correlation analysis
top30_genes = X_log.var(axis=0).sort_values(ascending=False).head(30).index
corr_matrix = X_log[top30_genes].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    cmap='coolwarm',
    center=0,
    linewidths=0.4,
    xticklabels=False,
    yticklabels=False,
    cbar_kws={'label': 'Pearson Correlation'}
)
plt.title('Pairwise Correlation Heatmap — Top 30 Variable Genes')
plt.tight_layout()
plt.show()

# Count highly correlated pairs (|r| > 0.9)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = (upper.abs() > 0.9).sum().sum()
print(f'Number of highly correlated gene pairs (|r| > 0.9): {high_corr_pairs}')


**Observation:**  
The correlation heatmap shows that some of the top variable genes are highly correlated. The printed value tells us how many gene pairs have `|r| > 0.9`. This means that some genes may carry similar information, so feature selection is useful to reduce redundancy.


---
## 3. Data Cleaning and Preprocessing

The preprocessing steps are:

1. Check missing values and duplicates.
2. Convert gene expression values to numeric values.
3. Fill missing values using the median of each gene.
4. Detect and assess outliers using Z-score analysis.
5. Apply `log1p` transformation.
6. Use stratified train/test split.
7. Apply feature selection using training data only.
8. Apply scaling using training data only.

These steps help prepare the data and avoid **data leakage**, which happens when information from the test set is used during training.


### 3.1 Data Quality Checks

In [ ]:
X_numeric_check = X.apply(pd.to_numeric, errors="coerce")

print("Missing values:", X_numeric_check.isna().sum().sum())
print("Duplicate rows:", X.duplicated().sum())
print("Negative values:", int((X_numeric_check.fillna(0).values < 0).sum()))

# If duplicates exist, we can remove them carefully while keeping labels aligned.
# In this dataset, we check first before changing anything.
if X.duplicated().sum() > 0:
    combined = X.copy()
    combined["Label"] = y.values
    combined = combined.drop_duplicates()
    y = combined["Label"].reset_index(drop=True)
    X = combined.drop(columns=["Label"]).reset_index(drop=True)
    print("Duplicates removed. New shape:", X.shape)
else:
    print("No duplicate samples were found.")


### 3.2 Log Transformation

In [ ]:
# Convert all gene columns to numeric values
X_numeric = X.apply(pd.to_numeric, errors="coerce")

# Check missing values before cleaning
print("Missing values before cleaning:", X_numeric.isna().sum().sum())

# Fill missing values with the median of each gene
X_clean = X_numeric.fillna(X_numeric.median())

# If any NaN values remain, fill them with 0
X_clean = X_clean.fillna(0)

print("Missing values after cleaning:", X_clean.isna().sum().sum())

# Apply log1p transformation
X_log = np.log1p(X_clean)

print("\nBefore log1p:")
print(
    "Min:", round(np.nanmin(X_clean.values), 3),
    "Max:", round(np.nanmax(X_clean.values), 3),
    "Mean:", round(np.nanmean(X_clean.values), 3)
)

print("\nAfter log1p:")
print(
    "Min:", round(np.nanmin(X_log.values), 3),
    "Max:", round(np.nanmax(X_log.values), 3),
    "Mean:", round(np.nanmean(X_log.values), 3)
)


### 3.3 Outlier Detection

RNA-seq data can contain extreme expression values in some samples.  
We use the **Z-score method** to detect samples with abnormally high or low  
overall expression levels. Samples with many extreme gene values may indicate  
technical artifacts or corrupted measurements.


In [ ]:
from scipy import stats

# Compute Z-scores across the log-transformed matrix
z_scores = np.abs(stats.zscore(X_log, axis=0))

# Flag genes with |z| > 3 per sample
outlier_gene_counts = (z_scores > 3).sum(axis=1)  # per sample
outlier_sample_counts = (z_scores > 3).sum(axis=0)  # per gene

print('Per-sample extreme gene counts (|Z| > 3):')
print(f'  Max:    {outlier_gene_counts.max()}')
print(f'  Mean:   {outlier_gene_counts.mean():.2f}')
print(f'  Median: {np.median(outlier_gene_counts):.1f}')

# Flag samples where more than 5% of genes are outliers
threshold_pct = 0.05
flagged = (outlier_gene_counts > threshold_pct * X_log.shape[1]).sum()
print(f'\nSamples where >5% of genes are outliers: {flagged}')

# Visualize per-sample outlier gene counts
plt.figure(figsize=(7, 4))
plt.hist(outlier_gene_counts, bins=40, color='steelblue', edgecolor='white')
plt.axvline(threshold_pct * X_log.shape[1], color='red', linestyle='--',
            label=f'5% threshold ({int(threshold_pct * X_log.shape[1])} genes)')
plt.title('Per-Sample Outlier Gene Count (|Z| > 3)')
plt.xlabel('Number of Outlier Genes per Sample')
plt.ylabel('Number of Samples')
plt.legend()
plt.show()


**Observation:**  
After `log1p` transformation, the extreme values became less severe. No sample had more than 5% of its genes flagged as outliers, so we did not remove any samples. This keeps the dataset size as large as possible for training.


### 3.4 Stratified Train/Test Split

In [ ]:
# Make sure X_log and y have the same index before splitting
X_log = X_log.reset_index(drop=True)
y = y.reset_index(drop=True)

# Keep only the rows that exist in both X_log and y
min_len = min(len(X_log), len(y))
X_log = X_log.iloc[:min_len].copy()
y = y.iloc[:min_len].copy()

print("X_log shape:", X_log.shape)
print("y shape:", y.shape)

# Train/test split using stratify because the classes are imbalanced
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_log,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set shape:", X_train_raw.shape)
print("Testing set shape:", X_test_raw.shape)

print("\nTraining class distribution:")
display(y_train.value_counts().sort_index())

print("\nTesting class distribution:")
display(y_test.value_counts().sort_index())


**Why stratified split?**  
Before splitting the data, we reset the index of the features and labels to make sure they are aligned. We used stratified splitting because the dataset is imbalanced, so the train and test sets keep a similar class distribution.


---
## 4. Feature Selection and Scaling

Since the dataset has thousands of genes and only a few hundred samples, using all features may cause overfitting.  
We use two feature selection steps:

1. **VarianceThreshold:** Removes genes that do not vary across samples.
2. **SelectKBest with ANOVA F-test:** Keeps the genes most associated with the target label.

Feature selection and scaling are applied only after the train/test split. This is important because the test set should stay unseen during training.


In [ ]:
# Step 1: Remove constant / near-constant genes using only the training set
var_filter = VarianceThreshold(threshold=0.0)

X_train_var = var_filter.fit_transform(X_train_raw)
X_test_var = var_filter.transform(X_test_raw)

var_genes = X_train_raw.columns[var_filter.get_support()]

print("Genes before variance filter:", X_train_raw.shape[1])
print("Genes after variance filter:", X_train_var.shape[1])

# Step 2: Select top K genes using ANOVA F-test
K_BEST = min(1000, X_train_var.shape[1])

selector = SelectKBest(score_func=f_classif, k=K_BEST)
X_train_selected = selector.fit_transform(X_train_var, y_train)
X_test_selected = selector.transform(X_test_var)

selected_genes = var_genes[selector.get_support()]

print("Selected genes:", X_train_selected.shape[1])
print("Example selected genes:", list(selected_genes[:10]))

# Step 3: Scale selected features for models that need scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

print("Scaling completed using StandardScaler.")
print("Train scaled mean:", round(X_train_scaled.mean(), 4))
print("Train scaled std:", round(X_train_scaled.std(), 4))

---
## 5. Model Design and Training

We train three main models from different categories, as required by the project.  
We also add Decision Tree as a simple extra baseline model.

| Category | Model | Reason |
|---|---|---|
| Traditional ML | Logistic Regression | Strong baseline for binary classification and interpretable coefficients |
| Ensemble Model | Random Forest | Captures non-linear patterns and provides feature importance |
| Deep Learning Model | MLP Neural Network | Learns non-linear combinations of gene expression features |
| Extra Traditional ML | Decision Tree | Simple and easy to interpret, useful for comparison |

We selected these models because they represent different machine learning categories required in the project. Logistic Regression is a simple traditional baseline, Random Forest is an ensemble model that can learn non-linear patterns, and MLP is a deep learning model. Decision Tree was added as an extra simple baseline for comparison.

Because the dataset is imbalanced, we use `class_weight='balanced'` where supported.


### 5.1 Traditional ML Model: Logistic Regression

In [ ]:
# Logistic Regression is a traditional ML model suitable for binary classification.
# class_weight='balanced' helps reduce bias toward the majority class.
log_reg = LogisticRegression(
    max_iter=3000,
    solver="liblinear",
    class_weight="balanced",
    random_state=RANDOM_STATE
)

log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression training completed.")

### 5.2 Extra Traditional ML Model: Decision Tree

Decision Tree was added as an extra traditional machine learning model.  
It is not required as one of the three main models, but it is useful because it is simple, interpretable, and easy to compare with Random Forest.


In [ ]:
# Trees do not require scaling, so we use the selected features directly.
dt_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

dt_model.fit(X_train_selected, y_train)

y_pred_dt = dt_model.predict(X_test_selected)
y_prob_dt = dt_model.predict_proba(X_test_selected)[:, 1]

print("Decision Tree Extra model training completed.")


### 5.3 Ensemble Model: Random Forest

In [ ]:
# Random Forest is an ensemble of decision trees.
# It can model non-linear relationships and rank genes by importance.
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train_selected, y_train)

y_pred_rf = rf.predict(X_test_selected)
y_prob_rf = rf.predict_proba(X_test_selected)[:, 1]

print("Random Forest training completed.")

### 5.4 Deep Learning Model: MLP Neural Network

In [ ]:
# MLP is a feed-forward neural network.
# Architecture: 1000 input features -> 128 hidden neurons -> 64 hidden neurons -> output class.
# early_stopping=True helps reduce overfitting.
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=RANDOM_STATE
)

mlp.fit(X_train_scaled, y_train)

y_pred_mlp = mlp.predict(X_test_scaled)
y_prob_mlp = mlp.predict_proba(X_test_scaled)[:, 1]

print("MLP Neural Network training completed.")

---
## 6. Model Evaluation and Comparison

Since the dataset is imbalanced, we compare models using several metrics:

- **Accuracy:** Overall correct predictions.
- **Precision:** How many predicted tumor/normal samples were correct.
- **Recall:** How many actual tumor/normal samples were found.
- **F1-score:** Balance between precision and recall.
- **Macro F1:** Treats both classes equally, useful for imbalanced data.
- **ROC-AUC:** Measures ranking ability across thresholds.

In [ ]:
def evaluate_model(model_name, y_true, y_pred, y_prob):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (Macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall (Macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 (Macro)": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, y_pred_lr, y_prob_lr),
    evaluate_model("Decision Tree (Extra)", y_test, y_pred_dt, y_prob_dt),
    evaluate_model("Random Forest", y_test, y_pred_rf, y_prob_rf),
    evaluate_model("MLP Neural Network", y_test, y_pred_mlp, y_prob_mlp)
])

results_sorted = results.sort_values(by="F1 (Macro)", ascending=False).reset_index(drop=True)
display(results_sorted)


**Evaluation note:**  
The models were evaluated using accuracy, precision, recall, F1-score, macro F1, and ROC-AUC. Since the dataset is imbalanced, macro F1 is very important because it gives attention to both normal and tumor classes instead of only the majority class.


### 6.1 Classification Reports

In [ ]:
for name, pred in [
    ("Logistic Regression", y_pred_lr),
    ("Decision Tree (Extra)", y_pred_dt),
    ("Random Forest", y_pred_rf),
    ("MLP Neural Network", y_pred_mlp)
]:
    print("\n" + "="*70)
    print(name)
    print("="*70)
    print(classification_report(y_test, pred, target_names=["Normal", "Tumor"], zero_division=0))


### 6.2 Confusion Matrices

In [ ]:
models_for_cm = {
    "Logistic Regression": y_pred_lr,
    "Decision Tree (Bonus)": y_pred_dt,
    "Random Forest": y_pred_rf,
    "MLP Neural Network": y_pred_mlp
}

for model_name, pred in models_for_cm.items():
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Tumor"])
    disp.plot()
    plt.title(f"Confusion Matrix — {model_name}")
    plt.show()


**Confusion matrix note:**  
The confusion matrices show the number of correct and incorrect predictions for each class. This helps us check whether a model is biased toward the tumor class or if it can also detect normal samples correctly.


### 6.3 ROC Curves

In [ ]:
plt.figure(figsize=(7, 5))

RocCurveDisplay.from_predictions(y_test, y_prob_lr, name="Logistic Regression")
RocCurveDisplay.from_predictions(y_test, y_prob_dt, name="Decision Tree (Extra)")
RocCurveDisplay.from_predictions(y_test, y_prob_rf, name="Random Forest")
RocCurveDisplay.from_predictions(y_test, y_prob_mlp, name="MLP Neural Network")

plt.title("ROC Curves")
plt.show()


### 6.4 Strengths and Weaknesses

| Model | Strengths | Weaknesses |
|---|---|---|
| Logistic Regression | Simple, fast, interpretable, good for high-dimensional data | Mostly linear, may miss complex relationships |
| Decision Tree (Extra) | Very easy to understand and explain | Can overfit and may be less stable than ensemble methods |
| Random Forest | Handles non-linear patterns and gives feature importance | Can overfit if not tuned, less direct than logistic regression |
| MLP Neural Network | Can learn complex non-linear patterns | Needs more data, can overfit, less interpretable |

**Main evaluation focus:** Macro F1 and confusion matrix, because the normal class is much smaller than the tumor class.


---
## 7. Model Optimization

To improve performance and meet the optimization requirement, we apply:

1. **Hyperparameter tuning** using cross-validation.
2. **Regularization tuning** for Logistic Regression.
3. **Random Forest parameter tuning**.
4. **Feature selection comparison** to test different numbers of selected genes.

### 7.1 Feature Selection Comparison

**Optimization plan:**  
For optimization, we tested different numbers of selected genes and tuned model parameters using GridSearchCV. The goal was to improve macro F1 and reduce overfitting, especially because the dataset has many features and a small number of samples.


In [ ]:
# Compare different numbers of selected genes using Logistic Regression.
# This helps us justify the chosen feature count.

k_values = [100, 300, 500, 1000, 2000]
k_values = [k for k in k_values if k <= X_train_var.shape[1]]

feature_results = []

for k in k_values:
    temp_selector = SelectKBest(score_func=f_classif, k=k)
    Xtr_k = temp_selector.fit_transform(X_train_var, y_train)
    Xte_k = temp_selector.transform(X_test_var)

    temp_scaler = StandardScaler()
    Xtr_k_scaled = temp_scaler.fit_transform(Xtr_k)
    Xte_k_scaled = temp_scaler.transform(Xte_k)

    temp_lr = LogisticRegression(
        max_iter=3000,
        solver="liblinear",
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
    temp_lr.fit(Xtr_k_scaled, y_train)

    pred_k = temp_lr.predict(Xte_k_scaled)
    prob_k = temp_lr.predict_proba(Xte_k_scaled)[:, 1]

    feature_results.append({
        "K Features": k,
        "F1 (Macro)": f1_score(y_test, pred_k, average="macro", zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob_k)
    })

feature_results_df = pd.DataFrame(feature_results)
display(feature_results_df)

plt.figure(figsize=(7, 4))
sns.lineplot(data=feature_results_df, x="K Features", y="F1 (Macro)", marker="o")
plt.title("Feature Selection Comparison")
plt.ylabel("Macro F1 Score")
plt.show()

### 7.2 Optimize Logistic Regression

In [ ]:
# Tune regularization strength C.
# Smaller C = stronger regularization.
param_grid_lr = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_lr = GridSearchCV(
    estimator=LogisticRegression(
        max_iter=3000,
        solver="liblinear",
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    param_grid=param_grid_lr,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1
)

grid_lr.fit(X_train_scaled, y_train)

best_lr = grid_lr.best_estimator_
y_pred_lr_opt = best_lr.predict(X_test_scaled)
y_prob_lr_opt = best_lr.predict_proba(X_test_scaled)[:, 1]

print("Best Logistic Regression parameters:", grid_lr.best_params_)
print("Best CV Macro F1:", round(grid_lr.best_score_, 4))

### 7.3 Optimize Random Forest

In [ ]:
param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [8, 12, None],
    "min_samples_leaf": [1, 2, 4]
}

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    param_grid=param_grid_rf,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1
)

grid_rf.fit(X_train_selected, y_train)

best_rf = grid_rf.best_estimator_
y_pred_rf_opt = best_rf.predict(X_test_selected)
y_prob_rf_opt = best_rf.predict_proba(X_test_selected)[:, 1]

print("Best Random Forest parameters:", grid_rf.best_params_)
print("Best CV Macro F1:", round(grid_rf.best_score_, 4))

### 7.4 Optimize MLP Neural Network

We tune the MLP architecture and regularization parameter (`alpha`) using `GridSearchCV`  
with stratified 3-fold cross-validation. The `alpha` parameter controls L2 regularization,  
which helps prevent overfitting on this small-sample, high-dimensional dataset.


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_mlp = {
    'hidden_layer_sizes': [(64,), (128, 64), (256, 128)],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.0005]
}

grid_mlp = GridSearchCV(
    estimator=MLPClassifier(
        activation='relu',
        solver='adam',
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=RANDOM_STATE
    ),
    param_grid=param_grid_mlp,
    scoring='f1_macro',
    cv=3,
    n_jobs=-1
)

grid_mlp.fit(X_train_scaled, y_train)

best_mlp = grid_mlp.best_estimator_
y_pred_mlp_opt = best_mlp.predict(X_test_scaled)
y_prob_mlp_opt = best_mlp.predict_proba(X_test_scaled)[:, 1]

print('Best MLP parameters:', grid_mlp.best_params_)
print('Best CV Macro F1:', round(grid_mlp.best_score_, 4))


### 7.5 Before vs After Optimization


In [ ]:
optimized_results = pd.DataFrame([
    evaluate_model("Logistic Regression , Baseline", y_test, y_pred_lr, y_prob_lr),
    evaluate_model("Logistic Regression , Optimized", y_test, y_pred_lr_opt, y_prob_lr_opt),
    evaluate_model("Decision Tree  , Extra", y_test, y_pred_dt, y_prob_dt),
    evaluate_model("Random Forest , Baseline", y_test, y_pred_rf, y_prob_rf),
    evaluate_model("Random Forest , Optimized", y_test, y_pred_rf_opt, y_prob_rf_opt),
    evaluate_model("MLP Neural Network , Baseline", y_test, y_pred_mlp, y_prob_mlp),
    evaluate_model("MLP Neural Network , Optimized", y_test, y_pred_mlp_opt, y_prob_mlp_opt),
])

optimized_results = optimized_results.sort_values(by="F1 (Macro)", ascending=False).reset_index(drop=True)
display(optimized_results)

plt.figure(figsize=(10, 5))
sns.barplot(data=optimized_results, x="F1 (Macro)", y="Model")
plt.title("Model Performance — Baseline vs Optimized")
plt.xlabel("Macro F1 Score")
plt.ylabel("Model")
plt.show()


In [ ]:
# Identify the best model based on Macro F1 score
best_model_row = optimized_results.sort_values(by="F1 (Macro)", ascending=False).iloc[0]
print("Best model based on Macro F1:", best_model_row["Model"])
print("Best Macro F1:", round(best_model_row["F1 (Macro)"], 4))
print("ROC-AUC:", round(best_model_row["ROC-AUC"], 4))


**Optimization Interpretation:**  
The optimization results compare the baseline models with the tuned models. If the optimized version has a higher macro F1 score, this means the tuning improved the model. If the improvement is small, it means the baseline model was already strong or the dataset size limited further improvement.

Based on the results table, the best model should be the one with the highest Macro F1 score and a balanced confusion matrix.


---
## 8. Feature Importance and Gene Ranking

To make the project more interpretable, we identify the genes that contributed most to the best models.

Important note: these are **model-based rankings**, not confirmed clinical biomarkers.

### 8.1 Logistic Regression Important Genes

In [ ]:
lr_importance = pd.DataFrame({
    "Gene": selected_genes,
    "Coefficient": best_lr.coef_[0],
    "Absolute Importance": np.abs(best_lr.coef_[0])
}).sort_values(by="Absolute Importance", ascending=False)

display(lr_importance.head(20))

plt.figure(figsize=(9, 6))
sns.barplot(data=lr_importance.head(20), x="Absolute Importance", y="Gene")
plt.title("Top 20 Important Genes — Logistic Regression")
plt.xlabel("Absolute Coefficient Value")
plt.ylabel("Gene")
plt.show()

### 8.2 Random Forest Important Genes

In [ ]:
rf_importance = pd.DataFrame({
    "Gene": selected_genes,
    "Importance": best_rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

display(rf_importance.head(20))

plt.figure(figsize=(9, 6))
sns.barplot(data=rf_importance.head(20), x="Importance", y="Gene")
plt.title("Top 20 Important Genes — Random Forest")
plt.xlabel("Feature Importance")
plt.ylabel("Gene")
plt.show()

**Interpretation note:**  
Feature importance helps us understand which selected genes contributed most to the model decisions. For Logistic Regression, we used the absolute coefficient values. For Random Forest, we used feature importance scores. These genes are useful for model interpretation, but they are not confirmed clinical biomarkers.


### Final Model Choice

Based on the test results, the Optimized Random Forest was the best model. It achieved Macro F1 = 1.0 and ROC-AUC = 1.0.

We selected Macro F1 as an important metric because the dataset is imbalanced, so it checks the performance on both Normal and Tumor classes. The strong result suggests that the selected gene features separated the two classes clearly in our dataset.

However, since the dataset size is limited, this result should be interpreted carefully and can be validated on more data in the future.

Overall, this project showed that data cleaning, log transformation, feature selection, and model tuning are important steps when working with high-dimensional gene expression data.
